# 04 - ML Forecast Features (Gold)

**Grain:** one row per province, crop, year and quarter. Used to train the ML model.

- `production_next_quarter` is the **label** (what the model predicts). It is not a feature.
- The label is only filled when the next row is really the next quarter (`time_index` + 1).
  The latest quarter of each series has no label; those rows are used later for forward forecasts.
- **Missing lags are filled with the last known production** of the same province + crop, so a series with a
  missing quarter does not lose its history. Only the very first row of a series stays `NULL`.
- `production_year_ago` keeps the strict value (`NULL` when that quarter is missing). It is **not** a model
  feature; batch inference uses it for the predicted YoY growth.
- `growth_rate_yoy` and `risk_label` are left out on purpose (target leakage).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from common.delta_io import write_delta_table
from configs.etl_config import SILVER_CROP_WEATHER_TABLE, GOLD_FORECAST_FEATURES_TABLE


enriched_df = spark.table(SILVER_CROP_WEATHER_TABLE)

by_province_and_crop = Window.partitionBy("province", "crop").orderBy("time_index")
next_time_index = F.lead("time_index", 1).over(by_province_and_crop)
next_production = F.lead("production", 1).over(by_province_and_crop)

# Last production seen in any EARLIER row of the same series
earlier_rows = by_province_and_crop.rowsBetween(Window.unboundedPreceding, -1)
last_known_production = F.last("production", ignorenulls=True).over(earlier_rows)

features_df = (
    enriched_df
    .withColumn(
        "production_next_quarter",
        F.when(next_time_index - F.col("time_index") == 1, next_production),
    )
    # Strict "same quarter as the label, one year earlier" value (3 quarters back), before any filling
    .withColumn("production_year_ago", F.col("production_lag_3"))
)

for quarters_back in (1, 2, 3, 4):
    lag_column = f"production_lag_{quarters_back}"
    features_df = features_df.withColumn(lag_column, F.coalesce(F.col(lag_column), last_known_production))

features_df = (
    features_df
    .select(
        "province",
        "crop",
        "crop_group",
        "year",
        "quarter",
        "quarter_num",
        "time_index",
        "production",
        "production_next_quarter",
        "production_lag_1",
        "production_lag_2",
        "production_lag_3",
        "production_lag_4",
        "production_year_ago",
        "temperature_2m_mean",
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum",
        "rain_sum",
        "precipitation_hours",
        "sunshine_duration",
        "reference_evapotranspiration_mm",
        "wind_speed_10m_max",
        "wind_gusts_10m_max",
        "rainfall_deviation_pct",
        "temperature_anomaly_c",
        "temp_anomaly_rolling",
        "rain_anomaly_rolling",
        "oni_index",
        "enso_phase_la_nina",
        "enso_phase_neutral",
    )
)

write_delta_table(features_df, GOLD_FORECAST_FEATURES_TABLE)

output_df = spark.table(GOLD_FORECAST_FEATURES_TABLE)
print(f"Rows: {output_df.count():,}")
print(f"Rows with a next-quarter label: {output_df.filter(F.col('production_next_quarter').isNotNull()).count():,}")